In [37]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import eigs
from scipy.linalg import eigh
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
def compute_z_laplacian(A, feature_matrix, num_components=9, penalty_factor=0.5):
    """
    Computes the Z-Laplacian matrix from the adjacency matrix A and feature matrix, applying penalties
    based on eigenvalue localization and feature similarity.

    Parameters:
    A : np.ndarray
        Adjacency matrix of the graph (n x n)
    feature_matrix : np.ndarray
        Feature matrix (n x d), where n is the number of nodes and d is the feature dimension.
    num_components : int
        Number of primary components (largest eigenvalues) to apply stronger penalties to.
    penalty_factor : float
        Factor by which to penalize the largest eigenvalues (0 < penalty_factor < 1).

    Returns:
    Lz : np.ndarray
        The Z-Laplacian matrix (n x n).
    """
    
    # Step 1: Eigen decomposition of adjacency matrix A
    eigenvalues, eigenvectors = eigh(A)
    num_components = A.shape[0]
    
    # Step 2: Calculate the feature similarity matrix (cosine similarity)
    # This matrix represents how similar the features are between different nodes
    feature_similarity = cosine_similarity(feature_matrix)
    
    # Normalize the feature similarity matrix to have values between 0 and 1
    feature_similarity = (feature_similarity - feature_similarity.min()) / (feature_similarity.max() - feature_similarity.min())
    
    # Step 3: Penalize the eigenvalues based on the feature similarity
    regularized_eigenvalues = eigenvalues.copy()
    
    # Use the feature similarity to penalize eigenvalues differently for each component
    for i in range(num_components):
        # Compute an average penalty factor based on feature similarity of the nodes
        penalty = penalty_factor * np.mean(feature_similarity[i])
        regularized_eigenvalues[i] *= penalty  # Apply penalty to the top i largest eigenvalues
    
    # Step 4: Construct the Z matrix as a diagonal matrix using regularized eigenvalues
    Z = np.diag(regularized_eigenvalues)
    
    # Step 5: Compute the Z-Laplacian as Lz = A + Z
    Lz = A + Z
    
    return Lz

In [51]:
# Function to initialize P randomly
def initialize_P(n, k):
    P = np.random.rand(n, k)
    P /= np.sum(P, axis=1)[:, np.newaxis]
    return P

# Function to compute the regularized Laplacian (Lz)
def compute_regularized_laplacian(A, Z):
    return A + Z

# Non-backtracking matrix-based method to estimate number of communities
def estimate_num_communities(A, features, initial_k=2, step=5, max_k=20):
    # Step 1: Calculate the degrees of the nodes
    degrees = np.sum(A, axis=1)
    diag_degrees = diags(degrees)
    
    # Step 2: Construct the non-backtracking matrix
    non_backtracking_matrix = np.block([
        [np.zeros_like(A), diag_degrees - np.eye(len(A))],
        [-np.eye(len(A)), A]
    ])
    
    # Step 3: Compute the norm of the non-backtracking matrix
   # norm_non_backtracking = norm(non_backtracking_matrix,'fro')  # Frobenius norm
    radius = np.sqrt(np.sum(np.abs(non_backtracking_matrix.data)**2))

    k = initial_k
    communities_count = 0
    all_real_found = False

    while not all_real_found and k <= max_k:
    # while k <= max_k:
        # Step 4: Compute the eigenvalues of the non-backtracking matrix
        eigvals, _ = eigs(non_backtracking_matrix, k=initial_k, which='LM')
        # print(non_backtracking_matrix)
        print(eigvals)
        print(radius)
        
        # Step 5: Filter out eigenvalues inside the circle with radius equal to the norm of the matrix
        eigen_distances = np.abs(eigvals)  # Magnitude of each eigenvalue
        outside_circle = eigen_distances > radius
        
        # Count the number of eigenvalues outside the circle
        communities_count = np.sum(outside_circle)
        
        if communities_count > 0:
           all_real_found = True  # Stop if we have enough eigenvalues to count
        
        else:
            # Increase k and try again
            k += step

    return communities_count

# Symmetric NMF update rule
def update_P(Lz, P, beta=0.5):
    numerator = np.dot(Lz, P)
    denominator = np.dot(np.dot(P, P.T), P)
    denominator[denominator == 0] = 1e-10  # Avoid division by zero
    P = P * ((1 - beta) + beta * numerator / denominator)
    return np.maximum(P, 1e-10)  # Ensure non-negativity

# CDNMF algorithm
def CDNMF(A, features, num_iterations=100, beta=0.5):
    n = A.shape[0]
    
    # Step 1: Estimate the number of communities (k) using spectral analysis
    # k = estimate_num_communities(A, features, initial_k=5, step=5, max_k=20)
    k = 7  # For simplicity, we will use a fixed number of communities
    print(f"Estimated number of communities: {k}")
    
    # Step 2: Initialize P matrix randomly
    P = initialize_P(n, k)
    
    # # Step 3: Initialize regularization matrix Z
    # IPR = np.zeros(n)  # Store IPR for each node
    # for i in range(n):
    #     IPR[i] = np.sum(features[i] ** 4)
    
    # Z = np.diag(1 + beta * IPR)
    
    # Step 4: Compute regularized Laplacian Lz
    Lz = compute_z_laplacian(A, features, num_components=9, penalty_factor=0.5)
    
    # Step 5: Iteratively update P using the update rule
    for iteration in range(num_iterations):
        P = update_P(Lz, P, beta)
        
        # Normalize P so that each row sums to 1
        P /= np.sum(P, axis=1)[:, np.newaxis]

        # Compute the objective function value
        obj_value = np.linalg.norm(Lz - np.dot(P, P.T), 'fro')

        # If the objective value is below a certain threshold, stop the iterations
        if obj_value < 1e-6:
            break
    
    # Assign nodes to communities based on maximum membership value
    communities = np.argmax(P, axis=1)
    
    return communities, P


In [52]:
# Function to parse .cites and .content files and build the adjacency matrix
def parse_dataset(cites_file, content_file):
    # Parse the .content file
    features = []
    node_id_to_index = {}
    num_nodes = 0
    with open(content_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            node_id = parts[0]
            node_id_to_index[node_id] = num_nodes
            feature_vector = np.array([float(x) for x in parts[1:-1]])
            features.append(feature_vector)
            num_nodes += 1

    # Parse the .cites file
    edges = []
    with open(cites_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            edges.append((parts[0], parts[1]))
    
    print(f"Number of nodes: {num_nodes}")
    print(f"Number of edges: {len(edges)}")
    print(f"Feature dimension: {len(features[0])}")
    
    # Build the adjacency matrix
    A = np.zeros((num_nodes, num_nodes))
    for edge in edges:
        if edge[0] not in node_id_to_index or edge[1] not in node_id_to_index:
            continue
        x = node_id_to_index[edge[0]]
        y = node_id_to_index[edge[1]]
        A[x, y] = 1
        # print(f"Adding edge between {x} and {y}")
    
    return A, features

In [53]:
def calculate_modularity(A, communities):
    """
    Calculate modularity of a graph with adjacency matrix A and community assignments.
    
    A: numpy array (adjacency matrix)
    communities: list or array of community assignments for each node
    """
    num_nodes = A.shape[0]
    m = np.sum(A)  # Total number of edges in the graph
    modularity = 0.0
    
    # Calculate degree of each node
    degrees = np.sum(A, axis=1)
    
    for i in range(num_nodes):
        for j in range(num_nodes):
            if communities[i] == communities[j]:  # Only consider nodes in the same community
                modularity += A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
    
    return modularity / (2 * m)

In [54]:
# Example usage
if __name__ == "__main__":
    # Load the dataset
    cites_file = "Datasets/cora/cora.cites"
    content_file = "Datasets/cora/cora.content"
    A, features = parse_dataset(cites_file, content_file)

    # Run the CDNMF algorithm
    communities, P = CDNMF(A, features, num_iterations=100, beta=0.999)
    
    # Output community assignment
    print("Community assignment:", communities)
    # print("Community membership matrix P:\n", P)

    # Calculate modularity
    modularity = calculate_modularity(A, communities)
    print(f"Modularity: {modularity}")

Number of nodes: 2708
Number of edges: 5429
Feature dimension: 1433
Estimated number of communities: 7
Community assignment: [0 4 0 ... 5 2 3]
Modularity: 0.19771609362943487
